# MVCA latent：整夜任务执行器（runner v1）

1. Runtime → Change runtime type → **A100 GPU + High-RAM**。
2. Runtime → **Run all**，允许挂载 Drive。第二格会一直运行（这是正常的），它会自动执行 Claude 放进 `MVCA_latent/runner/jobs_pending/` 的任务。
3. 标签页保持打开，电脑插电不要休眠。要停止：点停止按钮即可。

In [ ]:
# ── 1. 环境 ────────────────────────────────────────────────────────────────
import os, sys, time, json, shutil, traceback, subprocess, gc, re, math, glob, itertools
!pip install -q -U "transformers>=4.51" accelerate huggingface_hub bitsandbytes scikit-learn sentence-transformers
!pip install -q --no-deps git+https://github.com/anthropics/jacobian-lens.git
import torch, numpy as np, pandas as pd, torch.nn.functional as F
from google.colab import userdata, drive
from huggingface_hub import login, whoami
_tok = userdata.get("HF_TOKEN"); login(token=_tok, add_to_git_credential=False)
drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive/MVCA_latent/runner"
for d in ("jobs_pending", "jobs_running", "jobs_done", "jobs_failed", "results", "materials"):
    os.makedirs(f"{ROOT}/{d}", exist_ok=True)
GPU = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9
print(GPU, f"| {VRAM:.0f} GB | HF:", whoami(token=_tok)["name"], "| disk free", shutil.disk_usage("/content").free // 10**9, "GB")
import hashlib
_lib = open(f"{ROOT}/runner_lib.py", "rb").read()
assert hashlib.sha256(_lib).hexdigest() == "5ecfa41b3b4a6e84e2a43aeeac7928a1d749023ad7781da8fecd36daccd7997e", "runner_lib.py 校验不符：请告诉 Claude"
exec(compile(_lib.decode("utf-8"), "runner_lib.py", "exec"), globals())

In [ ]:
# ── 2. 执行器主循环（会一直运行；这是正常的）
run_loop()